# Notebook 03 — Multi-Head Attention

## Learning Objectives
- Understand *why* multiple heads: each head can focus on a different aspect of context.
- Use `MultiHeadAttention` from `src.models.attention`.
- Trace the split-and-merge head mechanism: `[batch, seq, d_model]` → heads → back.
- Visualize all 4 attention heads in parallel on the same input.
- Compare how different heads attend to different positions.

## Big Picture
Single-head attention compresses all relationships into one d_model-wide channel.
Multi-head attention splits d_model into H independent subspaces (d_k = d_model / H each),
runs scaled dot-product attention in each, then concatenates the results.
Each head can specialize: one might track syntax, another semantics, another coreference.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_table_csv
from src.utils.paths import RUNS_ATTENTION_DIR
from src.models.attention import MultiHeadAttention

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_ATTENTION_DIR, clean=False)
print(f"Device: {device}")
print("All imports successful.")

## 2. Dataset: Synthetic Token Embeddings

We create fake token embeddings for a short sequence to trace the MHA computation.
The input shape `[batch, seq_len, d_model]` is the universal currency of Transformer layers.

In [ ]:
# Hyperparameters
BATCH_SIZE = 2
SEQ_LEN    = 8
D_MODEL    = 64
NUM_HEADS  = 4
D_K        = D_MODEL // NUM_HEADS  # = 16 per head

torch.manual_seed(42)

# Input: a batch of token embeddings [batch, seq_len, d_model]
x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL).to(device)

print(f"Input shape     : {x.shape}  (batch, seq_len, d_model)")
print(f"d_model         : {D_MODEL}")
print(f"num_heads       : {NUM_HEADS}")
print(f"d_k per head    : {D_K}  (d_model / num_heads)")
print(f"Total params/head: Q+K+V = 3 × {D_MODEL}×{D_K} = {3*D_MODEL*D_K}")

## 3. Architecture (Text Diagram)

```
Input [batch, seq, d_model]
        │
   ┌────┼─────────────────┐
   │    │  Linear Q,K,V   │   (3 weight matrices: d_model → d_model)
   └────┼─────────────────┘
        │
   Split into H=4 heads
   [batch, H, seq, d_k]     d_k = d_model / H = 16
        │
   ┌────┬────┬────┬────┐
   │Head│Head│Head│Head│   ScaledDotProduct per head
   │ 0  │ 1  │ 2  │ 3  │
   └────┴────┴────┴────┘
        │
   Concatenate heads
   [batch, seq, H × d_k] = [batch, seq, d_model]
        │
   Linear output projection  (d_model → d_model)
        │
   Output [batch, seq, d_model]
```

## 4. Theory: Why Multiple Heads?

**Single head** can only compute one set of query-key similarity patterns.

**Multi-head** projects Q, K, V into H lower-dimensional subspaces and runs
attention independently in each. Because the projections are learned separately,
different heads can attend to:
- **Head 0**: recent context (local patterns)
- **Head 1**: subject-verb agreement  
- **Head 2**: coreference (who does "it" refer to?)
- **Head 3**: rare long-distance dependencies

Parameter count: 4 matrices of d_model × d_model for Q, K, V, O.
This is the same as 1 large head — but multi-head gives much richer representations.

## 5. Implementation: Using MultiHeadAttention

In [ ]:
# Instantiate Multi-Head Attention
mha = MultiHeadAttention(d_model=D_MODEL, num_heads=NUM_HEADS, dropout=0.0).to(device)
print(mha)
n_params = sum(p.numel() for p in mha.parameters())
print(f"\nTotal parameters: {n_params:,}")
print(f"  w_q: {mha.w_q.weight.shape}")
print(f"  w_k: {mha.w_k.weight.shape}")
print(f"  w_v: {mha.w_v.weight.shape}")
print(f"  w_o: {mha.w_o.weight.shape}")

In [ ]:
# Forward pass: self-attention (Q = K = V = x)
mha.eval()
with torch.no_grad():
    output, attn_weights = mha(
        query=x,
        key=x,
        value=x,
        mask=None,
        return_weights=True,
    )

print(f"Input  shape  : {x.shape}      (batch, seq_len, d_model)")
print(f"Output shape  : {output.shape}  (batch, seq_len, d_model)")
print(f"Weights shape : {attn_weights.shape}  (batch, num_heads, seq_q, seq_k)")
print(f"\nWeights sum per query (batch=0, head=0): {attn_weights[0, 0].sum(dim=-1)}")

In [ ]:
# Inspect each head individually
print("Per-head attention statistics (batch=0):")
print("-" * 60)
for h in range(NUM_HEADS):
    w = attn_weights[0, h].detach().cpu()  # [seq_q, seq_k]
    entropy = -(w * (w + 1e-9).log()).sum(dim=-1).mean().item()
    max_w = w.max().item()
    argmax_col = w.argmax(dim=-1)  # which position each query attends most to
    diag_mean = w.diag().mean().item()  # self-attention tendency
    print(f"  Head {h}: entropy={entropy:.3f}  max_weight={max_w:.3f}  "
          f"self-attn={diag_mean:.3f}")

## 6. Sanity Checks

In [ ]:
# Shape checks
assert output.shape == (BATCH_SIZE, SEQ_LEN, D_MODEL), \
    f"Wrong output shape: {output.shape}"
assert attn_weights.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN), \
    f"Wrong weights shape: {attn_weights.shape}"
print("Shape checks passed.")

# Weights sum to 1 across key dimension
row_sums = attn_weights.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), \
    "Attention weights don't sum to 1!"
print("Attention weight sum-to-1 check passed (all heads).")

# d_k check
assert mha.d_k == D_K, f"Expected d_k={D_K}, got {mha.d_k}"
print(f"d_k check passed: d_k = d_model / num_heads = {D_MODEL} / {NUM_HEADS} = {mha.d_k}")

# No NaN
assert not torch.isnan(output).any()
print("No NaN values. All checks passed!")

## 7. Visualization: One Heatmap Per Head

In [ ]:
# Plot one heatmap per head (2×2 grid for 4 heads)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

tok_labels = [f'tok{i}' for i in range(SEQ_LEN)]

for h in range(NUM_HEADS):
    ax = axes[h]
    w = attn_weights[0, h].detach().cpu().numpy()  # [seq_q, seq_k]
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    ax.set_title(f'Head {h}  (max={w.max():.3f})', fontsize=12)
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(tok_labels, fontsize=8)
    ax.set_yticklabels(tok_labels, fontsize=8)
    ax.set_xlabel('Key position (attended to)')
    ax.set_ylabel('Query position (attending)')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f'Multi-Head Attention Weights\n'
    f'd_model={D_MODEL}, num_heads={NUM_HEADS}, d_k={D_K}, seq_len={SEQ_LEN}',
    fontsize=13
)
fig.tight_layout()

plot_path = RUNS_ATTENTION_DIR / 'multi_head_attention_heads.png'
fig.savefig(plot_path, dpi=120)
plt.close(fig)
print(f"Multi-head heatmap saved to: {plot_path}")

In [ ]:
# Save shape information to CSV
shape_rows = [
    {'tensor': 'input x',       'shape': str(tuple(x.shape)),            'description': 'Token embeddings [batch, seq, d_model]'},
    {'tensor': 'output',        'shape': str(tuple(output.shape)),        'description': 'MHA output [batch, seq, d_model]'},
    {'tensor': 'attn_weights',  'shape': str(tuple(attn_weights.shape)),  'description': '[batch, heads, seq_q, seq_k]'},
    {'tensor': 'Q (projected)', 'shape': f'({BATCH_SIZE}, {NUM_HEADS}, {SEQ_LEN}, {D_K})', 'description': 'After split_heads'},
    {'tensor': 'K (projected)', 'shape': f'({BATCH_SIZE}, {NUM_HEADS}, {SEQ_LEN}, {D_K})', 'description': 'After split_heads'},
    {'tensor': 'V (projected)', 'shape': f'({BATCH_SIZE}, {NUM_HEADS}, {SEQ_LEN}, {D_K})', 'description': 'After split_heads'},
    {'tensor': 'w_q weight',    'shape': str(tuple(mha.w_q.weight.shape)), 'description': 'Q projection weight'},
    {'tensor': 'w_o weight',    'shape': str(tuple(mha.w_o.weight.shape)), 'description': 'Output projection weight'},
]
csv_path = RUNS_ATTENTION_DIR / 'multi_head_shapes.csv'
save_table_csv(shape_rows, csv_path)
print(f"Shape info saved to: {csv_path}")
for row in shape_rows:
    print(f"  {row['tensor']:<20} {row['shape']:<40} {row['description']}")

## 8. Failure Cases

**d_model not divisible by num_heads**: The `MultiHeadAttention` constructor raises an `AssertionError`. Always check d_model % num_heads == 0.

**All heads attending to the same position**: With random initialization, heads may not differentiate. After training, diverse patterns emerge because the loss penalizes uniformity.

**Very small d_k**: With 1 head and d_model=8, d_k=8. With 8 heads, d_k=1 — a single scalar! Information bottleneck becomes severe. Typical practice: d_k ≥ 32.

**return_weights=False (default)**: The module returns `(output, None)`. Don't forget `return_weights=True` when you need to visualize attention.

## 9. Exercises

1. **num_heads=1**: Run MHA with num_heads=1. How does the single-head attention map compare to the 4-head version?
2. **Cross-attention**: Use a different sequence for Q (length 6) and K,V (length 8). Verify the output shape is `[batch, 6, d_model]`.
3. **Parameter count**: Calculate analytically why the total parameter count is `4 × d_model²` (Q, K, V, O projections). Verify against `sum(p.numel() for p in mha.parameters())`.
4. **Head diversity**: After 10 random seeds, which heads consistently have highest/lowest entropy? Why might different initializations produce different head specializations?
5. **Masked MHA**: Apply a causal mask (from Notebook 02) to the MHA forward pass. Verify the upper-triangular portion of all heads is zero.

## 10. Key Takeaways

- **Multi-head attention** = H parallel scaled dot-product attention modules on d_k = d_model/H subspaces.
- **Linear projections** (w_q, w_k, w_v, w_o) let each head learn what to query, what to attend to, and what to extract.
- **Split–merge pattern**: `[batch, seq, d_model]` → `[batch, H, seq, d_k]` → attention → `[batch, seq, d_model]`.
- Different heads specialize on different patterns after training (syntax, semantics, position, etc.).
- Parameter count = 4 × d_model² (independent of num_heads) — heads are a free diversity boost.